# Chương 7: Deep Graph & Mạng Nơ-ron Đồ Thị (GNN / GCN)
**Học viện Công nghệ Bưu chính Viễn thông (PTIT)**
*Môn học: Phát triển các Hệ thống Thông minh (GVHD: PGS.TS Trần Đình Quế)*
*Sinh viên: Phạm Văn Tư - B23DCCN879*

## 1. Toán Tử Chuẩn Hóa Laplacian Kipf & Welling Bằng NumPy
Thực hiện Renormalization trick và phép nhân ma trận đối xứng:

In [1]:
import numpy as np

# Tạo đồ thị đồ chơi 5 nút
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 1, 1],
    [0, 1, 1, 0, 1],
    [0, 0, 1, 1, 0]
], dtype=float)

N = A.shape[0]
A_tilde = A + np.eye(N)
D_tilde = np.sum(A_tilde, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(D_tilde))
norm_adj = D_inv_sqrt @ A_tilde @ D_inv_sqrt

print("Ma trận kề chuẩn hóa đối xứng Kipf:")
print(np.round(norm_adj, 3))


Ma trận kề chuẩn hóa đối xứng Kipf:
[[0.333 0.289 0.258 0.    0.   ]
 [0.289 0.25  0.224 0.25  0.   ]
 [0.258 0.224 0.2   0.224 0.258]
 [0.    0.25  0.224 0.25  0.289]
 [0.    0.    0.258 0.289 0.333]]


## 2. Xây Dựng Mạng GCN 2 Tầng Bằng PyTorch & Huấn Luyện Trên Đồ Thị:

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.W)

    def forward(self, x, adj):
        return torch.mm(adj, torch.mm(x, self.W))

class GCN(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super(GCN, self).__init__()
        self.gc1 = GCNLayer(num_features, hidden_dim)
        self.gc2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, adj):
        h = F.relu(self.gc1(x, adj))
        h = F.dropout(h, p=0.5, training=self.training)
        out = self.gc2(h, adj)
        return F.log_softmax(out, dim=1)

# Giả lập đồ thị Cora (500 nút, 1433 đặc trưng, 7 lớp)
x_feat = torch.randn(500, 1433)
adj_tensor = torch.eye(500) + 0.05 * torch.rand(500, 500) # Mock normalized adj
y_labels = torch.randint(0, 7, (500,))

model_gcn = GCN(num_features=1433, hidden_dim=64, num_classes=7)
optimizer = torch.optim.Adam(model_gcn.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(1, 101):
    model_gcn.train()
    optimizer.zero_grad()
    out = model_gcn(x_feat, adj_tensor)
    loss = F.nll_loss(out, y_labels)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        pred = out.argmax(dim=1)
        acc = (pred == y_labels).float().mean() * 100
        print(f"Epoch {epoch:3d}/100 - Loss: {loss.item():.4f} - Train Acc: {acc:.2f}%")


Epoch  20/100 - Loss: 1.9445 - Train Acc: 15.40%
Epoch  40/100 - Loss: 1.9459 - Train Acc: 15.20%
Epoch  60/100 - Loss: 1.9459 - Train Acc: 15.20%
Epoch  80/100 - Loss: 1.9459 - Train Acc: 15.20%
Epoch 100/100 - Loss: 1.9459 - Train Acc: 15.20%
